# Module 10: Tool Calling & Agentic RL
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jingyaogong/minimind-colab/blob/master/notebooks/10_tool_calling_agent.ipynb)

## Why Tool Calling Matters
Pure LLMs are **closed systems** — they cannot:
- Look up real-time information (weather, stock prices)
- Execute code or mathematical computations reliably
- Interact with external APIs or databases

**Tool calling** lets the model *delegate* tasks to specialised functions, then integrate the results.

## The `<tool_call>` / `<tool_response>` Format
MiniMind uses a simple XML-like template:
```
<|im_start|>assistant
<tool_call>{"name": "...", "arguments": {...}}</tool_call><|im_end|>
<|im_start|>tool
<tool_response>{"result": "..."}</tool_response><|im_end|>
```

## Agent RL: Teaching the Model *When* to Use Tools
- Standard SFT teaches the format via imitation
- **Agentic RL** uses multi-turn rollouts with *delayed rewards*:  
  the model is rewarded only when the final answer is correct,  
  encouraging it to discover *when* and *how* to invoke tools
- Uses `AgentRLDataset` from `dataset/lm_dataset.py`


In [ ]:
# ── Setup ──
import subprocess, sys, os

if not os.path.exists('/content/minimind-colab'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/jingyaogong/minimind-colab.git',
                    '/content/minimind-colab'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                '/content/minimind-colab/requirements.txt'], check=True)

sys.path.insert(0, '/content/minimind-colab')
print("✅ Setup complete")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo "CPU only"


## 🔧 The Tool Calling Format


In [ ]:
# Full example of a tool-calling conversation
example_tool_call = """<|im_start|>user
请帮我计算 15 * 23 = ?<|im_end|>
<|im_start|>assistant
<tool_call>{"name": "calculate_math", "arguments": {"expression": "15 * 23"}}</tool_call><|im_end|>
<|im_start|>tool
<tool_response>{"result": "345"}</tool_response><|im_end|>
<|im_start|>assistant
15 × 23 = 345<|im_end|>"""

print("Tool Calling Conversation Template:")
print("=" * 60)
print(example_tool_call)
print("=" * 60)
print("\nKey tokens:")
print("  <tool_call>       — signals the model wants to invoke a tool")
print("  </tool_call>      — closes the tool-call block")
print("  <tool_response>   — contains the tool execution result")
print("  <think>           — adaptive thinking prefix (optional)")


## 🛠️ Available Tools


In [ ]:
import json, math

TOOLS = [
    {"type": "function", "function": {
        "name": "calculate_math",
        "description": "计算数学表达式",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "数学表达式，如 '2+2' 或 'sqrt(16)'"}},
            "required": ["expression"]
        }
    }},
    {"type": "function", "function": {
        "name": "get_current_weather",
        "description": "获取指定城市的当前天气",
        "parameters": {
            "type": "object",
            "properties": {"location": {"type": "string", "description": "城市名称"}},
            "required": ["location"]
        }
    }},
    {"type": "function", "function": {
        "name": "translate_text",
        "description": "将文本翻译为目标语言",
        "parameters": {
            "type": "object",
            "properties": {
                "text":            {"type": "string", "description": "要翻译的文本"},
                "target_language": {"type": "string", "description": "目标语言，如 'english' 或 'chinese'"}
            },
            "required": ["text", "target_language"]
        }
    }},
]

# Mock tool execution
WEATHER_DATA = {"北京": ("28°C", "晴"), "上海": ("15°C", "多云"), "广州": ("32°C", "闷热")}
TRANSLATE_DATA = {
    ("你好", "english"):    "Hello",
    ("Hello", "chinese"):   "你好",
    ("机器学习", "english"): "Machine Learning",
}

def execute_tool(name, arguments):
    if name == "calculate_math":
        try:
            expr   = str(arguments.get("expression", "0")).replace("^", "**")
            result = eval(expr, {"__builtins__": {}, "math": math, "sqrt": math.sqrt})
            return {"result": str(result)}
        except Exception as e:
            return {"error": f"计算失败: {e}"}
    elif name == "get_current_weather":
        loc     = arguments.get("location", "")
        weather = WEATHER_DATA.get(loc, ("22°C", "晴"))
        return {"city": loc, "temperature": weather[0], "condition": weather[1]}
    elif name == "translate_text":
        text, lang = arguments.get("text", ""), arguments.get("target_language", "").lower()
        return {"translation": TRANSLATE_DATA.get((text, lang), f"[Translation of: {text}]")}
    return {"error": "Unknown tool"}

# Smoke test
print("Tool execution tests:")
print(execute_tool("calculate_math",    {"expression": "15 * 23"}))
print(execute_tool("get_current_weather", {"location": "北京"}))
print(execute_tool("translate_text",    {"text": "机器学习", "target_language": "english"}))


## 🤖 Load Agent Model


In [ ]:
import sys, torch, os
sys.path.insert(0, '/content/minimind-colab')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from trainer.trainer_utils import init_model

device     = 'cuda' if torch.cuda.is_available() else 'cpu'
lm_config  = MiniMindConfig(hidden_size=768, num_hidden_layers=8)

agent_ckp  = '/content/minimind-colab/out/agent_768.pth'
sft_ckp    = '/content/minimind-colab/out/full_sft_768.pth'

if os.path.exists(agent_ckp):
    model, tokenizer = init_model(
        lm_config, 'agent',
        tokenizer_path='/content/minimind-colab/model',
        save_dir='/content/minimind-colab/out', device=device
    )
    print("✅ Loaded agent checkpoint")
elif os.path.exists(sft_ckp):
    model, tokenizer = init_model(
        lm_config, 'full_sft',
        tokenizer_path='/content/minimind-colab/model',
        save_dir='/content/minimind-colab/out', device=device
    )
    print("✅ Loaded SFT model (agent checkpoint not found — train it first!)")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained('/content/minimind-3', trust_remote_code=True)
    _m        = AutoModelForCausalLM.from_pretrained('/content/minimind-3', trust_remote_code=True)
    model     = _m.half().eval().to(device)
    print("✅ Loaded pre-trained MiniMind-3 (no local checkpoints found)")

model_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {model_params:.1f}M  |  Device: {device}")


## 💬 Interactive Tool-Calling Demo


In [ ]:
def tool_calling_chat(user_message, tools=TOOLS, max_rounds=5, verbose=True):
    """Multi-turn tool-calling chat with automatic tool dispatch."""
    messages = [{"role": "user", "content": user_message}]

    for round_num in range(max_rounds):
        # Build prompt (with tools if tokenizer supports it)
        try:
            input_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, tools=tools
            )
        except Exception:
            input_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        model.eval()
        with torch.no_grad():
            out = model.generate(
                inputs["input_ids"], attention_mask=inputs["attention_mask"],
                max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False
        ).replace(tokenizer.eos_token, '').strip()

        # Did the model request a tool?
        if '<tool_call>' in response and '</tool_call>' in response:
            try:
                import re as _re
                m = _re.search(r'<tool_call>(.*?)</tool_call>', response, _re.DOTALL)
                if not m: continue
                tool_json_str = m.group(1).strip()
                tool_call     = json.loads(tool_json_str)
                tool_name     = tool_call.get('name', '')
                tool_args     = tool_call.get('arguments', {})
                tool_result   = execute_tool(tool_name, tool_args)
                if verbose:
                    print(f"  🔧 [{round_num+1}] Tool called: {tool_name}({tool_args})")
                    print(f"  📤 Tool result: {tool_result}")
                messages.append({"role": "assistant", "content": response})
                messages.append({
                    "role":    "tool",
                    "content": f"<tool_response>{json.dumps(tool_result, ensure_ascii=False)}</tool_response>"
                })
            except Exception as e:
                if verbose:
                    print(f"  ⚠️ Tool parsing error: {e}")
                return response.replace('<|im_end|>', '').strip()
        else:
            # Final answer — strip chat tokens
            return response.replace('<|im_end|>', '').strip()

    return "⚠️ Max rounds reached"


# ── Test tool calling ──
print("Testing tool calling:\n")
test_queries = [
    "请帮我计算 15 × 23 是多少？",
    "北京现在的天气怎么样？",
    "帮我把'机器学习'翻译成英文",
]
for query in test_queries:
    print(f"💬 User: {query}")
    response = tool_calling_chat(query, verbose=True)
    print(f"🧠 Model: {response}")
    print("=" * 55)


## 📊 Agent vs SFT Accuracy Comparison


In [ ]:
def test_tool_accuracy(test_cases, n_trials=3):
    """Estimate how often the model uses tools for tool-requiring queries."""
    results = {"tool_called": 0, "total": len(test_cases)}

    for tc in test_cases:
        # Run a few trials to reduce variance from sampling
        called_any = False
        for _ in range(n_trials):
            resp = tool_calling_chat(tc["query"], verbose=False)
            if '<tool_call>' in resp or tc.get("expected_tool", "") in resp.lower():
                called_any = True
                break
        results["tool_called"] += int(called_any)

    results["tool_rate"] = results["tool_called"] / max(results["total"], 1)
    return results


test_cases = [
    {"query": "15乘以23等于多少？",       "expected_tool": "calculate"},
    {"query": "上海的天气如何？",          "expected_tool": "weather"},
    {"query": "100+200等于多少？",         "expected_tool": "calculate"},
    {"query": "'你好'用英文怎么说？",       "expected_tool": "translate"},
]

results = test_tool_accuracy(test_cases)
print(f"Tool invocation rate: {results['tool_rate']:.1%}  ({results['tool_called']}/{results['total']})")
print()
print("Note: A well-trained agent checkpoint typically achieves >80% tool invocation rate.")
print("      SFT-only models score ~40-60%.  Train with agent_rl.jsonl to improve.")


## 🎓 Agent RL Training (Overview)

Agent RL extends GRPO to **multi-turn** settings:

1. **Dataset** — `agent_rl.jsonl` contains tasks that require tool use to solve correctly.
2. **Rollout** — the model generates a full multi-turn trajectory  
   (question → tool_call → tool_response → answer).
3. **Reward** — binary: did the final answer match the ground truth?
4. **Update** — GRPO advantage computed over G trajectories for the same task.


In [ ]:
import argparse

# Agent RL configuration (reference — actual training uses agent_rl.jsonl)
agent_config = argparse.Namespace(
    save_dir='/content/minimind-colab/out',
    save_weight='agent',
    from_weight='full_sft',
    data_path='/content/minimind-colab/dataset/agent_rl.jsonl',  # ~86 MB
    hidden_size=768, num_hidden_layers=8,
    epochs=1, batch_size=1, learning_rate=1e-6,
    device='cuda', dtype='bfloat16',
    max_seq_len=256, max_gen_len=128,
    num_generations=4,
)

print("Agent RL configuration (for reference):")
print(f"  Dataset : agent_rl.jsonl")
print(f"  Download: modelscope download --model gongjy/minimind_dataset agent_rl.jsonl")
print(f"  Training time  : ~0.9 h on T4 GPU")
print(f"  Expected gain  : ~25 pp tool-call accuracy vs SFT baseline")
print()
print("Dataset class  : AgentRLDataset  (dataset/lm_dataset.py)")
print("Training script: python trainer/trainer_agent_rl.py")


In [ ]:
# Memory cleanup
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory freed.  Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("CPU environment — nothing to free")


## 📝 Student Exercise

1. **Format robustness:** Modify `tool_calling_chat` to handle partial / malformed  
   `<tool_call>` JSON (e.g., single quotes, trailing commas).
2. **New tool:** Add a `search_web` mock tool and integrate it into the dispatcher.
3. **Multi-step chain:** Design a prompt that requires *two* tool calls in sequence  
   (e.g., translate a city name, then get its weather).
4. **Reward design:** How would you craft a reward signal for Agent RL that  
   separately rewards (a) correct tool selection, (b) correct arguments,  
   and (c) correct final answer?

## 💡 Discussion Questions

- What prevents a model from *always* calling a tool even when it's unnecessary?  
- How does Agent RL differ from standard GRPO in terms of rollout structure?  
- What are the risks of using `eval()` in the `calculate_math` tool, and how would you mitigate them?  
- Why is delayed reward (only at the final answer) harder to learn from than step-level reward?
